# 3. Climate stress test

WF3 **generation** and WF4 **simulation and retained metrics**.

<div style="border-left:4px solid #999;padding:0.5em 0.9em;background:#f6f6f6">
<b>Source only &mdash; outputs are not committed.</b><br>
Run the notebook to see results, or read a rendered copy published as an
Artifact. Outputs are cleared on commit
(<code>dev/scripts/notebook_outputs.py --strip</code>) and their absence is
enforced by <code>tests/test_notebook_outputs.py</code> on both CI legs.<br>
Why: a notebook carrying figures embeds them as base64 and does not
delta-compress, so every edit added megabytes to history permanently &mdash;
these three were 8.8 MB, and one rename sweep that rewrote three short
strings inside them cost a 7.1 MB push.
</div>

## What these workflows do

`generate_scenarios.smk` perturbs locally generated weather and publishes a
reusable collection. The simulation runner consumes that collection and the
model from notebook 1, retains native responses, and publishes metric sets.
Generation has no model dependency. CMIP6 remains a terminal plausibility overlay.
The unperturbed generated runs define reference membership; they are outside the
perturbation surface and retain their distinct processing history.


## Setup

In [ ]:
# Standard library, plus the few third-party names the results cells need.
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython import display

In [ ]:
# Work from the repository root.
#
# Snakemake resolves every path in a config relative to the directory it is
# invoked from, and the shipped configs are written against the repo root.
# Rather than hardcoding an install path -- which is what made the previous
# version of this notebook unrunnable for anyone but its author -- walk up from
# wherever the kernel started until the Snakefiles come into view.
def find_repo_root(marker: str = "build_model.smk") -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"could not find {marker} at or above {Path.cwd()}; "
        "start the kernel inside the blueearth_cst checkout"
    )


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(REPO_ROOT)

In [ ]:
# Snakemake runs take minutes, so stream the output rather than buffering it to
# the end of the cell.
#
# `check` raises on a nonzero exit. Without it a failed Snakemake call printed
# its error into the cell output and the cell still rendered as SUCCESSFUL, so
# the first symptom was a later `display.Image` raising on a file that was never
# written -- an error pointing at the wrong place, in a document whose reader is
# the least equipped to tell the difference. Pass `check=False` where a nonzero
# exit is expected, as `snakemake --unlock` gives when there is no lock.
def run(command: str, *, check: bool = True) -> int:
    print("$ " + command)
    print()
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        shell=True,
        stderr=subprocess.STDOUT,
        bufsize=1,
        close_fds=True,
    ) as process:
        for line in iter(process.stdout.readline, b""):
            print(line.rstrip().decode("utf-8", errors="replace"))
    if check and process.returncode != 0:
        raise RuntimeError(
            f"command failed with exit code {process.returncode}: {command}"
        )
    return process.returncode

## The settings files

The project file names separate generation and simulation files. Composition
loads their settings with `config_path` relative to the project file; ordinary
paths remain relative to the run directory.


In [ ]:
from blueearth_cst.shared.config_composition import compose_config

CONFIG = "test_case/project_config_rapid.yml"
raw = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))
cfg, config_paths = compose_config(raw, CONFIG)
PROJECT_DIR = Path(cfg["project"]["project_dir"])
PROJECT_NAME = cfg.get("project_name") or PROJECT_DIR.name
print("config:", CONFIG, "project:", PROJECT_DIR)


In [ ]:
GEN_CFG = cfg["workflows"]["generate_scenarios"]
EXP_CFG = cfg["workflows"]["simulate_system"]
# The shipped rapid config pins its experiment name. Pin one for this notebook
# when adapting a project so the result reader selects the same namespace.
EXPERIMENT = EXP_CFG["experiment_name"]
EXP_DIR = PROJECT_DIR / "experiments" / EXPERIMENT
print("experiment:", EXPERIMENT)


In [ ]:
# The settings file, in full. It is commented in place -- those comments are the
# documentation for each key, which is why this cell prints the shipped file
# rather than re-authoring a copy that could drift away from it.
print(Path(CONFIG).read_text(encoding="utf-8"))

### Generation and simulation settings

Generation owns realization count, simulation window, monthly perturbations,
generator configuration, seed and identifier capacity. Simulation owns the
experiment name, operation, optional explicit collection manifest, compute
controls and metric selection. Migrating an old config preserves its resolved
seed; new automatic seeds exclude experiment names and identifier capacity.

Changing simulation inputs requires a new experiment name. Changing metrics
creates a new immutable set from the retained responses.


## The job graph

In [ ]:
print('WF3 selects its generation DAG only after the owned launcher freezes a plan.')


In [ ]:
# Missing identity checkpoints produce a partial graph until execution resolves them.
print("DAG images:", PROJECT_DIR / "logs" / "dag")


### Stage boundaries

Generation resolves source content, claims a collection, generates its declared
rows and publishes its readiness marker last. Simulation freezes the model and
collection, prepares model forcing and publishes native responses. Metric
planning resolves shared references before reduction. See the maintained
[rule index](../../dev/reference/workflows/rule-index.md) for the numbered rules.


## Running

Run generation first, then simulation through its mandatory runner. Clear a
Snakemake lock only after confirming that no workflow is running in this checkout.


In [ ]:
run(f'python scripts/generate_scenarios.py --config "{CONFIG}" --project-dir "{PROJECT_DIR}" --cores 3 -- --dry-run')


In [ ]:
# The owned runner executes every enabled workflow in CONFIG.
run(f'python scripts/run_workflows.py --config "{CONFIG}" --project-dir "{PROJECT_DIR}" --cores 3')
run(f'python scripts/simulate_system.py --config "{CONFIG}" --target all --dry-run')


In [ ]:
run(f'python scripts/simulate_system.py --config "{CONFIG}" --target all --cores 3')


## Results

In [ ]:
# What the run wrote. Noisy internal branches are skipped so the shape of the
# tree stays readable.
def show_tree(root: Path, skip=(".snakemake", "_parts"), max_files=8):
    root = Path(root)
    for path, dirs, files in os.walk(root):
        dirs[:] = sorted(d for d in dirs if d not in skip)
        rel = Path(path).relative_to(root)
        print(root.name + "/" if str(rel) == "." else f"{root.name}/{rel}")
        shown = sorted(f for f in files if not f.endswith(".xml"))
        for name in shown[:max_files]:
            print("  - " + name)
        if len(shown) > max_files:
            print(f"  ... {len(shown) - max_files} more")


show_tree(EXP_DIR)

### The retained scenario design

Read collection membership from the frozen simulation record. The monthly lookup
has twelve rows per perturbed design point and no unperturbed row. Preserve IDs
as text, including empty keys; never reconstruct them from filenames.


In [ ]:
simulation = json.loads((EXP_DIR / "config/simulation.json").read_text())
COLLECTION_DIR = Path(simulation["collection"]["manifest_path"]).parent
scenarios = pd.read_csv(COLLECTION_DIR / "scenario_table.csv", dtype=str, keep_default_na=False)
lookup = pd.read_csv(COLLECTION_DIR / "stress_test_lookup.csv", dtype={"st_id": str})
print(f"{len(scenarios)} runs; {lookup['st_id'].nunique()} perturbed design points")
display.display(scenarios.head(), lookup.head(12))


**How to read this.** The grid should bracket the region of *interest*,
not the region the projections point at &mdash; that is what makes the result
scenario-neutral.

A grid that only covers the CMIP6 cloud from notebook 2 cannot tell you where the
system's threshold is if the threshold lies outside it. And a threshold outside
the currently projected range is still worth knowing about: the projections will
be revised, the system's response will not.

### The generated weather

The collection retains forcing descriptors, code/environment/source inventories,
and the portable preparation context. Inspect declared units, calendars and
coverage before interpreting model results. Historical generator plots are not
part of the successor workflow's terminal targets.


In [ ]:
collection = json.loads((COLLECTION_DIR / "collection.json").read_text())
display.display(pd.DataFrame([
    {"run_id": item["run_id"], "path": item["path"],
     "calendar": item["descriptor"]["source_calendar"],
     "start": item["descriptor"]["start"], "end": item["descriptor"]["end"]}
    for item in collection["forcing"]
]))


### The metrics

Resolve the exact current metric request; do not select the latest directory.
The reader validates the ready set, its expected keys and membership before
returning tables. Class A and C are per run; Class B belongs to bundles.


In [ ]:
from blueearth_cst.experiment.metric_plan import current_metric_request, verify_metric_plan, read_metric_set
from blueearth_cst.shared.indicator_tables import indicator_tables
from blueearth_cst.shared.snake_utils import resolve_water_year_start

tokens = list(EXP_CFG.get("metrics", indicator_tables(cfg["model"]["outvars"])))
anchor = "YS-" + resolve_water_year_start(cfg["climate"].get("water_year_start")).upper()
request = current_metric_request(EXP_DIR, tokens, anchor)
plan = verify_metric_plan(EXP_DIR, request)
METRIC_DIR = Path(plan["targets"]["manifest"]).parent
read_metric_set(EXP_DIR, METRIC_DIR / "metrics.json")
q = pd.read_csv(METRIC_DIR / "q_indicators.csv", dtype={"unit_id": str, "location": str})
units = pd.read_csv(METRIC_DIR / "unit_index.csv", dtype=str, keep_default_na=False)
print(f"{len(q)} rows, {q['metric'].nunique()} metrics, {q['location'].nunique()} locations")
q.head(10)


In [ ]:
sorted(q["metric"].unique())

Confirm complete membership before aggregating. Each bundle's members must
resolve to one design point; the empty key denotes the unperturbed group.


In [ ]:
joined = units.merge(scenarios[["run_id", "st_id"]], left_on="member_run_id",
                     right_on="run_id", validate="many_to_one", how="left", indicator=True)
assert (joined["_merge"] == "both").all(), "unresolved scenario membership"
assert (joined.groupby("unit_id")["st_id"].nunique() == 1).all(), "mixed design bundle"
unit_design = joined.groupby("unit_id")["st_id"].first()
assert set(q["unit_id"]) <= set(unit_design.index), "unresolved metric unit"
q.groupby("metric").size()


### Deriving the axes

The response surface needs one number per member per axis, and the lookup holds
twelve. Collapsing them is specified by **HM-7**
(`dev/reference/contracts/hydrological-model-seam.md`), and the code below is
transcribed from that text rather than imported from the toolbox &mdash;
deliberately, because the consumers that actually draw these surfaces are
outside this repository and must re-implement it exactly this way. If the cell
below is hard to write from the contract alone, the contract is the thing to
fix.

Two rules do the real work:

- **The default month set is the months that VARY across members**, not all
  twelve. On a uniform design those are the same thing. On a seasonal one they
  are not, and averaging in the months that were held constant is precisely the
  misreport this design removed: +30% imposed in JJA collapses to +7.6% over the
  year.
- **Exactly-equal values short-circuit.** A weighted mean of twelve identical
  numbers does not reliably return that number in floating point, and the
  difference lands in the last bit of an axis label.

In [ ]:
# --- HM-7, transcribed. Imports nothing from the toolbox. -------------------
MONTH_LENGTHS = {1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
                 7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31}
MONTH_INITIALS = "JFMAMJJASOND"


def month_classes(lookup, column):
    """Step 1 - varying months and held levels. The threshold is EXACT zero."""
    grouped = lookup.groupby("month")[column]
    span = grouped.max() - grouped.min()
    varying = [int(m) for m in span.index if span[m] > 0]
    held = {int(m): float(grouped.first()[m]) for m in span.index if span[m] == 0}
    return varying, held


def collapse(values):
    """Month-length weighted mean, with the exact-equality short-circuit."""
    if len(set(values.values())) == 1:
        return float(next(iter(values.values())))
    weight = sum(MONTH_LENGTHS[m] for m in values)
    return sum(MONTH_LENGTHS[m] * v for m, v in values.items()) / weight


def derive_axis(lookup, column, months=None):
    """Steps 2 and 3 - one value per member, plus the months it collapsed."""
    varying, _held = month_classes(lookup, column)
    # Step 2: nothing varies -> degenerate, and M defaults to all twelve.
    months = list(months or varying or MONTH_LENGTHS)
    sub = lookup[lookup["month"].isin(months)]
    values = {
        st: collapse(dict(zip(m["month"].astype(int), m[column].astype(float))))
        for st, m in sub.groupby("st_id")
    }
    return values, months


ABBREV = ("Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec")


def label_months(months):
    """All twelve -> `the year`; a contiguous CIRCULAR run -> initials (<= 3)
    or `<first>-<last>` (>= 4); otherwise a comma list."""
    months = sorted(months)
    if len(months) == 12:
        return "the year"
    for start in months:
        run = [((start - 1 + k) % 12) + 1 for k in range(len(months))]
        if set(run) == set(months):
            if len(months) <= 3:
                return "".join(MONTH_INITIALS[m - 1] for m in run)
            return f"{ABBREV[run[0] - 1]}–{ABBREV[run[-1] - 1]}"
    return ", ".join(ABBREV[m - 1] for m in months)


def caption(lookup, column, unit, months=None):
    """HM-7's caption. Cases 1-3 and the degenerate 4-4c; the `also vary`
    clause for months outside M is in the contract and omitted here."""
    varying, held = month_classes(lookup, column)
    _values, months = derive_axis(lookup, column, months)
    in_m = sorted(set(held) & set(months))
    if not varying:  # step 2 reached first: degenerate, so no leading phrase
        levels = {held[m] for m in in_m}
        if levels == {0.0}:
            text = "unchanged"
        elif len(levels) == 1:
            text = f"held at {levels.pop():+.3g}{unit}"
        else:
            mean = collapse({m: held[m] for m in in_m})
            text = f"held at declared monthly offsets (weighted mean {mean:+.3g}{unit})"
        return text if len(months) == 12 else f"{text} in {label_months(months)}"
    text = f"mean change over {label_months(months)}"
    outside = sorted(set(held) - set(months))
    levels = {held[m] for m in outside}
    if outside and levels == {0.0}:
        text += f"; {label_months(outside)} unchanged"
    elif len(levels) == 1:
        text += f"; {label_months(outside)} held at {levels.pop():+.3g}{unit}"
    elif levels:
        text += "; remaining months held at declared monthly offsets"
    return text


print("temp: ", caption(lookup, "temp_change", " \N{DEGREE SIGN}C"))
print("precip:", caption(lookup, "precip_change", "%"))

In [ ]:
METRIC = "q_annual_mean"

# Join the derived axes onto the indicator rows. WG-2 pins ONE st_id width per
# lookup, so the join key's width is read from the table rather than assumed.
WIDTH = sorted({len(s) for s in lookup["st_id"]})
assert len(WIDTH) == 1, f"the lookup mixes st_id widths {WIDTH}"
WIDTH = WIDTH[0]
BASELINE = ""

temp_axis, _ = derive_axis(lookup, "temp_change")
precip_axis, _ = derive_axis(lookup, "precip_change")

rows = q.copy()
rows["st_id"] = rows["unit_id"].map(unit_design)
assert rows["st_id"].notna().all()

# `st_0` is partitioned OUT rather than plotted: it is the baseline, and it has
# no row in the lookup, so it has no place on the surface.
members = rows[rows["st_id"] != BASELINE].copy()
assert set(members["st_id"]) == set(lookup["st_id"]), (
    "the indicator table and the lookup disagree about which members exist -- "
    "a short join draws a surface with holes in it rather than reporting one"
)
members["temp_change"] = members["st_id"].map(temp_axis)
members["precip_change"] = members["st_id"].map(precip_axis)

# The downstream-most location, chosen by largest unperturbed mean flow rather
# than by a hardcoded id, so this cell survives a change of basin.
baselines = rows[(rows["metric"] == METRIC) & (rows["st_id"] == BASELINE)]
LOCATION = baselines.groupby("location")["value"].mean().idxmax()

surface = (
    members[(members["metric"] == METRIC) & (members["location"] == LOCATION)]
    .groupby(["temp_change", "precip_change"])["value"]
    .mean()
    .unstack("precip_change")
    .sort_index(ascending=False)
)
baseline = baselines[baselines["location"] == LOCATION]["value"].mean()
print(f"{METRIC} at location {LOCATION}; baseline (st_0) = {baseline:.4g}")
print(f"x: {caption(lookup, 'precip_change', '%')}")
print(f"y: {caption(lookup, 'temp_change', ' \N{DEGREE SIGN}C')}")
(100 * (surface / baseline - 1)).round(1)

The table expresses mean response as percent change from the unperturbed
group, with temperature rows and precipitation columns. This normalization
does not erase the unperturbed group's different processing history. Response
surfaces describe vulnerability across the chosen grid; CMIP6 changes can be
overlaid afterward to assess plausibility.


## Next

Change generation settings to explore another collection; use a new experiment
for changed simulation inputs. For new metrics from the same native responses,
set `operation: metrics-only` and run the simulation runner with `--target simulations_and_indicators`.
See [retained handoffs](../wf3-retained-handoffs.md) for the artifact contracts.
